# 07 · Reachable junctions against NetworkX

**Question.** Does Fieldwork's `reachable_nodes` widget traverse a street network the way
it says it does?

The widget declares a cheapest-path traversal from one origin over the edge lengths the
network supplies, keeping every junction whose cost is at most the budget, where the
budget is the minutes multiplied by the speed. It measures no geometry and uses no
projection.

NetworkX is the reference for a specific reason. The case study this widget was extracted
from builds its isochrones with `nx.ego_graph(..., distance=...)`, so this compares
Fieldwork against the implementation the original analysis actually used, not against an
arbitrary second opinion.

The fixture is the 738-junction Soho network the John Snow worked example ships, at that
example's own parameters: 5 minutes at 75 metres per minute, outbound, from the junction
nearest the Broad Street pump.

**Criteria fixed before running.** The same set of reached junctions, exactly — a junction
either is or is not within the budget, so there is no tolerance to give. Every cost within
1 cm, which is the rounding Fieldwork applies to the number it publishes rather than an
allowance for disagreement. And every remaining budget within 1 cm of the budget less the
cost, because that attribute is what the corridor step downstream consumes.

In [ ]:
NEEDED = ["networkx"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_07_reachability import run

report = run(Path.cwd() / "fixtures" / "reachability-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"network: {report['networkNodes']} junctions, {report['networkEdges']} directed streets")
print(f"budget: {report['budgetMetres']} m")
print(f"reached: Fieldwork {report['fieldwork']['reached']}, NetworkX {report['reference']['reached']}")
print(f"junctions only NetworkX reached: {report['reachedOnlyByReference'] or 'none'}")
print(f"junctions only Fieldwork reached: {report['reachedOnlyByFieldwork'] or 'none'}")
print(f"worst cost difference      {report['worstCostDifferenceM']:.3e} m")
print(f"worst remaining difference {report['worstRemainingDifferenceM']:.3e} m")

## Where the residual comes from

The worst cost difference is 5 mm, and it is not an error in either traversal. Fieldwork
rounds the cost it reports to two decimal places, so half a centimetre is the largest
difference the published attribute can express. Both implementations settle the same
junctions in the same order at the same costs; one of them publishes fewer digits.

That rounding is not free. The corridor step truncates a street by `remainingM / lengthM`,
so a 5 mm error in the remaining budget moves the end of a corridor by up to 5 mm — which
is why the composed corridor and the `isochrone` monolith agree to 3.8 parts per million
rather than exactly.

## What agreement here does not mean

Both implementations traverse the lengths the fixture states. Neither measures the streets,
so neither can tell you whether those lengths are right — if the network is wrong, both are
wrong together and this check still passes.

Nor does it say the reached set is what anyone can walk in five minutes. The 375 m budget
is 5 minutes at 4.5 km/h, which the case study states as an assumption about pedestrians in
Soho in **1854**. Both implementations inherit that assumption identically, so agreement
here is agreement about arithmetic, not about walkability.

In [ ]:
# How far the result travels with the assumption: the same graph, the same origin, at
# speeds spanning a slow walk to a brisk one. The budget scales, and so does the answer.
import networkx as nx

fixture = json.loads((Path.cwd() / "fixtures" / "reachability-soho-001.json").read_text())
declared = fixture["declaredParameters"]
graph = nx.DiGraph()
for node in fixture["input"]["network"]["nodes"]:
    graph.add_node(node["id"])
for edge in fixture["input"]["network"]["edges"]:
    existing = graph.get_edge_data(edge["from"], edge["to"], default=None)
    if existing is None or edge["lengthM"] < existing["length"]:
        graph.add_edge(edge["from"], edge["to"], length=edge["lengthM"])

print(f"{'km/h':>6} {'m/min':>7} {'budget m':>9} {'junctions reached':>18}")
for kmh in (3.0, 4.0, 4.5, 5.0, 6.0):
    m_per_min = kmh * 1000 / 60
    budget = declared["minutes"] * m_per_min
    reached = nx.single_source_dijkstra_path_length(
        graph, declared["originNodeId"], cutoff=budget, weight="length")
    marker = "  <- the case study's assumption" if abs(kmh - 4.5) < 1e-9 else ""
    print(f"{kmh:6.1f} {m_per_min:7.1f} {budget:9.0f} {len(reached):18d}{marker}")